In [1]:
%pip install python-dotenv groq langchain-groq --quiet

import os
import getpass
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if not os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY not found in .env file.")
    os.environ["GROQ_API_KEY"] = getpass.getpass("Please enter your Groq API Key: ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00
GROQ_API_KEY not found in .env file.
Please enter your Groq API Key: ··········


In [2]:
MODEL_CONFIG = {
    "technical": {
        "system": "You are a rigorous Technical Support Expert. You provide precise, code-focused solutions. Always check for common syntax errors and explain the logic behind your fix.",
        "model": "llama-3.3-70b-versatile",
        "temperature": 0.4
    },
    "billing": {
        "system": "You are an empathetic Billing Expert. You focus on financial inquiries and policy-driven solutions. Be helpful and clear about refund timelines and subscription terms.",
        "model": "llama-3.3-70b-versatile",
        "temperature": 0.3
    },
    "general": {
        "system": "You are a friendly General Support Assistant. You handle casual inquiries and general questions with a helpful and polite tone.",
        "model": "llama-3.1-8b-instant",
        "temperature": 0.7
    },
    "tool_use": {
        "system": "You are a financial data assistant with access to live market tools.",
        "model": "llama-3.1-8b-instant",
        "temperature": 0.3
    }
}

print("Expert configurations ready")

Expert configurations ready


In [3]:
# BONUS: Tool-Use Expert — mock fetch for real-time data queries

MOCK_DATA = {
    "bitcoin":  {"symbol": "BTC", "price": "$67,420.00", "change_24h": "+2.3%"},
    "ethereum": {"symbol": "ETH", "price": "$3,512.00",  "change_24h": "-0.8%"},
    "solana":   {"symbol": "SOL", "price": "$172.50",    "change_24h": "+5.1%"},
}

def mock_fetch_crypto_price(query: str) -> str:
    """Simulates fetching live crypto data based on the user query."""
    query_lower = query.lower()
    for coin, data in MOCK_DATA.items():
        if coin in query_lower:
            return (
                f"[TOOL RESULT] {data['symbol']} Price: {data['price']} "
                f"(24h change: {data['change_24h']})"
            )
    return "[TOOL RESULT] Sorry, that cryptocurrency is not in our mock database."

def handle_tool_use(user_input: str) -> str:
    """Step 1: call the mock tool. Step 2: pass result to LLM for a natural response."""
    tool_result = mock_fetch_crypto_price(user_input)
    print(f"  [Tool called] → {tool_result}")

    tool_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
    tool_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a financial data assistant. You have access to real-time tool results. "
         "Use the provided tool result to answer the user's question naturally and concisely. "
         "Do not make up prices — only use what the tool returned."),
        ("human", "User question: {query}\n\nTool result: {tool_result}")
    ])
    chain = tool_prompt | tool_llm | StrOutputParser()
    return chain.invoke({"query": user_input, "tool_result": tool_result})

print("Tool-use expert ready ✓")

Tool-use expert ready ✓


In [4]:
# router
def route_prompt(user_input):
    router_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

    router_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a routing specialist. Classify the user query into one of these categories: "
         "'technical', 'billing', 'general', or 'tool_use'. "
         "Use 'tool_use' for any query asking for real-time data like cryptocurrency or stock prices. "
         "Return ONLY the category name as a single word."),
        ("human", "{query}")
    ])

    router_chain = router_prompt | router_llm | StrOutputParser()
    category = router_chain.invoke({"query": user_input}).strip().lower()
    return category

# Quick test
test_query = "How do I fix a NullPointerException in my Java code?"
print(f"Query: {test_query}")
print(f"Routed to: {route_prompt(test_query)}")

Query: How do I fix a NullPointerException in my Java code?
Routed to: technical


In [5]:
def process_request(user_input):
    category = route_prompt(user_input)

    if category not in MODEL_CONFIG:
        category = "general"

    print(f"--- Routing to: {category.upper()} Expert ---")

    # Delegate to tool-use handler
    if category == "tool_use":
        return handle_tool_use(user_input)

    config = MODEL_CONFIG[category]
    expert_llm = ChatGroq(
        model=config["model"],
        temperature=config["temperature"]
    )
    expert_prompt = ChatPromptTemplate.from_messages([
        ("system", config["system"]),
        ("human", "{query}")
    ])
    final_chain = expert_prompt | expert_llm | StrOutputParser()
    return final_chain.invoke({"query": user_input})


# --- Original examples ---
print("EXAMPLE 1:")
print(process_request("My python script is throwing an IndexError on line 5."))

print("\n" + "="*50 + "\n")

print("EXAMPLE 2:")
print(process_request("I was charged twice for my subscription this month."))

EXAMPLE 1:
--- Routing to: TECHNICAL Expert ---
To accurately diagnose and resolve the `IndexError` on line 5 of your Python script, I would need to see the code itself. However, I can guide you through a general approach to troubleshooting and fixing such errors.

### Understanding `IndexError`

An `IndexError` occurs when you try to access an element in a sequence (like a list, tuple, or string) using an index that is out of range. For example, if you have a list with 5 elements (indexed from 0 to 4), trying to access the element at index 5 would raise an `IndexError`.

### Steps to Troubleshoot

1. **Review the Line of Code**: Look at line 5 of your script and identify the operation that's causing the error. It's likely an indexing operation on a list, tuple, or string.

2. **Check Index Values**: Ensure that the index you're using to access the sequence is within the valid range. Remember, indexing starts at 0, so the last valid index of a sequence is always one less than its lengt

In [6]:
# --- Bonus: Tool-Use examples ---
print("BONUS EXAMPLE 1:")
print(process_request("What is the current price of Bitcoin?"))

print("\n" + "="*50 + "\n")

print("BONUS EXAMPLE 2:")
print(process_request("How much does Ethereum cost right now?"))

print("\n" + "="*50 + "\n")

print("BONUS EXAMPLE 3 (unknown coin):")
print(process_request("What is the price of Dogecoin right now?"))

BONUS EXAMPLE 1:
--- Routing to: TOOL_USE Expert ---
  [Tool called] → [TOOL RESULT] BTC Price: $67,420.00 (24h change: +2.3%)
The current price of Bitcoin is $67,420.00, with a 24-hour change of +2.3%.


BONUS EXAMPLE 2:
--- Routing to: TOOL_USE Expert ---
  [Tool called] → [TOOL RESULT] ETH Price: $3,512.00 (24h change: -0.8%)
Ethereum is currently priced at $3,512.00, with a 24-hour change of -0.8%.


BONUS EXAMPLE 3 (unknown coin):
--- Routing to: TOOL_USE Expert ---
  [Tool called] → [TOOL RESULT] Sorry, that cryptocurrency is not in our mock database.
I'm not able to find the current price of Dogecoin with the available tool results.
